# APIs públicas · Conocimiento y ocio

Más APIs públicas sin clave, esta vez de contenido: [Wikipedia](https://www.mediawiki.org/wiki/API:Main_page) (resúmenes y búsqueda), [Open Library](https://openlibrary.org/developers/api) (libros y autores), [GitHub](https://docs.github.com/rest) (repositorios) y [PokéAPI](https://pokeapi.co) (datos de un videojuego, con relaciones entre recursos y estructuras recursivas).

Necesita conexión a internet. Las salidas son las de la fecha de ejecución: cifras como las estrellas de un repositorio cambian con el tiempo, así que las comprobaciones se fijan en la forma de los datos.

## 1. Resúmenes de Wikipedia

Escribir `resumen(titulo)`, que pida a la API REST de la Wikipedia en español el resumen de un artículo, y mostrar para el Teide, Garajonay y Timanfaya la descripción corta, las coordenadas y la primera frase del extracto.

In [1]:
import re
from collections import Counter

import pandas as pd
import requests

sesion = requests.Session()
sesion.headers["User-Agent"] = "ejercicios-python/1.0"


def obtener_json(url, **params):
    respuesta = sesion.get(url, params=params, timeout=30)
    respuesta.raise_for_status()
    return respuesta.json()


def resumen(titulo):
    return obtener_json(f"https://es.wikipedia.org/api/rest_v1/page/summary/{titulo}")


for titulo in ["Teide", "Parque_nacional_de_Garajonay", "Parque_nacional_de_Timanfaya"]:
    r = resumen(titulo)
    primera = re.split(r"(?<=\.)\s", r["extract"], maxsplit=1)[0]
    coordenadas = r.get("coordinates", {})
    print(f"{r['title']} — {r.get('description', 'sin descripción')}")
    print(f"  ({coordenadas.get('lat', float('nan')):.3f}, {coordenadas.get('lon', float('nan')):.3f})  {primera}")
    assert r["lang"] == "es" and r["extract"]

Teide — volcán en la isla Tenerife y pico más alto de España (volcán en caldera)
  (28.273, -16.644)  El Teide es un volcán situado en la isla española de Tenerife, en el archipiélago de Canarias, frente a la costa noroccidental de África y sobre la placa africana.
Parque nacional de Garajonay — parque nacional en España
  (28.126, -17.237)  El parque nacional de Garajonay es un espacio natural protegido español que se extiende por más del 10 % de la superficie de la isla de La Gomera, en la comunidad autónoma de Canarias.


Parque nacional de Timanfaya — parque nacional en Lanzarote, España
  (29.011, -13.780)  El parque nacional de Timanfaya es un espacio natural protegido español situado en los municipios de Yaiza y Tinajo, en la isla de Lanzarote, comunidad autónoma de Canarias.


## 2. Buscar en Wikipedia

Usar la API de MediaWiki (`action=query`, `list=search`) para buscar artículos sobre «volcanes de Canarias» y mostrar los diez primeros títulos con su número de palabras. Limpiar el fragmento de texto de la coincidencia, que llega con etiquetas HTML.

In [2]:
busqueda = obtener_json(
    "https://es.wikipedia.org/w/api.php",
    action="query", list="search", srsearch="volcanes de Canarias", srlimit=10, format="json",
)
resultados = busqueda["query"]["search"]
print("Coincidencias totales:", busqueda["query"]["searchinfo"]["totalhits"])
for r in resultados:
    fragmento = re.sub(r"<[^>]+>", "", r["snippet"])
    print(f"{r['title']:<45} {r['wordcount']:>6} palabras | {fragmento[:60]}…")
assert len(resultados) == 10 and all("<" not in re.sub(r"<[^>]+>", "", r["snippet"]) for r in resultados)

Coincidencias totales: 672
Monumento natural de los Volcanes de Teneguía    167 palabras | monumento natural de los Volcanes de Teneguía es un espacio …
Volcán de Fuencaliente                           536 palabras | El volcán de Fuencaliente fue un volcán que entró en erupció…
Volcán de San Martín                              98 palabras | GR-131, un sendero de Gran Recorrido​ «Volcán Martín o Tigal…
Reserva natural especial del Chinyero            501 palabras | isla de Tenerife (Canarias, España).​ El volcán Chinyero est…
Teneguía                                         408 palabras | un volcán situado en el municipio de Fuencaliente en la isla…
Erupción volcánica de La Palma de 1971           775 palabras | erupción del volcán Teneguía fue una erupción volcánica ocur…
Erupción volcánica de La Palma de 1949           905 palabras | erupción del volcán de San Juan fue una erupción volcánica o…
Volcán de Tajogaite                             1481 palabras | Volcán de Tajogaite,​​es el

## 3. Las obras de Galdós en Open Library

Buscar en Open Library las obras de Benito Pérez Galdós pidiendo solo los campos necesarios (`fields`). Mostrar las diez con más ediciones y contar, por décadas, cuántas obras se publicaron por primera vez entre 1860 y 1920.

In [3]:
obras = obtener_json(
    "https://openlibrary.org/search.json",
    author="Benito Pérez Galdós", fields="title,first_publish_year,edition_count", limit=500,
)
libros = pd.DataFrame(obras["docs"]).dropna(subset=["first_publish_year"])
libros["first_publish_year"] = libros["first_publish_year"].astype(int)
print(obras["numFound"], "resultados;", len(libros), "con año de publicación")
print(libros.nlargest(10, "edition_count").to_string(index=False))

epoca = libros[libros["first_publish_year"].between(1860, 1920)]
decadas = Counter(epoca["first_publish_year"] // 10 * 10)
for decada in sorted(decadas):
    print(f"{decada}s {'█' * decadas[decada]} {decadas[decada]}")
assert libros["edition_count"].ge(1).all()

2825 resultados; 497 con año de publicación
 edition_count  first_publish_year                title
            44                1886        Doña Perfecta
            37                1883            Marianela
            34                1895             Nazaŕin
            33                1888                 Miau
            31                1887  Fortunata y Jacinta
            31                1882 Episodios nacionales
            24                1913         Misericordia
            22                1884        La de Bringas
            21                1884            Trafalgar
            20                2011        Doña Perfecta
1870s ███████ 7
1880s ████████████████████ 20
1890s ██████████████████████████████████ 34
1900s ████████████████████████████████████████████ 44
1910s ███████████ 11


## 4. Un repositorio de GitHub

Pedir los datos del repositorio `python/cpython` (estrellas, *forks*, lenguaje principal, fecha de creación y licencia), el reparto de bytes por lenguaje (`/languages`) en porcentaje y las cinco etiquetas (*tags*) más recientes.

In [4]:
repo = obtener_json("https://api.github.com/repos/python/cpython")
print(f"{repo['full_name']}: {repo['description']}")
print(f"  ★ {repo['stargazers_count']:,} | forks {repo['forks_count']:,} | {repo['language']} | creado el {repo['created_at'][:10]} | licencia {repo['license']['spdx_id']}")

lenguajes = pd.Series(obtener_json("https://api.github.com/repos/python/cpython/languages"))
porcentajes = (lenguajes / lenguajes.sum() * 100).round(1).head(6)
print(porcentajes.to_string(), "\n(% del código)")

etiquetas = obtener_json("https://api.github.com/repos/python/cpython/tags", per_page=5)
print("Últimas etiquetas:", [e["name"] for e in etiquetas])
assert repo["language"] == "Python" and len(etiquetas) == 5

python/cpython: The Python programming language
  ★ 78,483 | forks 36,567 | Python | creado el 2017-02-10 | licencia NOASSERTION


Python        63.4
C             34.8
M4             0.4
C++            0.4
HTML           0.3
JavaScript     0.1 
(% del código)


Últimas etiquetas: ['v3.16.0a2', 'v3.16.0a1', 'v3.15.2', 'v3.15.1', 'v3.15.0']


## 5. Los repositorios más populares de una organización

Listar los repositorios públicos de la organización `langchain-ai` (hasta 100 en una página), quedarse con los que no son *forks* ni están archivados y mostrar los cinco con más estrellas, con su lenguaje y la fecha de su última actualización.

In [5]:
repos = obtener_json("https://api.github.com/orgs/langchain-ai/repos", per_page=100, type="public")
tabla = pd.DataFrame(
    [{"repositorio": r["name"], "estrellas": r["stargazers_count"], "lenguaje": r["language"], "actualizado": r["pushed_at"][:10]}
     for r in repos if not r["fork"] and not r["archived"]]
)
print(len(repos), "repositorios en la página;", len(tabla), "propios y activos")
print(tabla.nlargest(5, "estrellas").to_string(index=False))
assert tabla["estrellas"].ge(0).all()

100 repositorios en la página; 50 propios y activos
          repositorio  estrellas         lenguaje actualizado
            langchain     149985           Python  2026-11-22
            langgraph      45269           Python  2026-11-22
          langchainjs      18357       TypeScript  2026-11-20
     rag-from-scratch      10139 Jupyter Notebook  2025-06-26
local-deep-researcher       9417           Python  2026-10-15


## 6. Comparar fichas de Pokémon

Descargar de PokéAPI las fichas de cuatro Pokémon iniciales y construir un `DataFrame` con sus tipos y sus seis estadísticas base (que vienen en una lista de diccionarios anidados). Añadir el total y ordenar de mayor a menor.

In [6]:
def estadisticas(nombre):
    datos = obtener_json(f"https://pokeapi.co/api/v2/pokemon/{nombre}")
    fila = {"nombre": datos["name"], "tipos": "/".join(t["type"]["name"] for t in datos["types"])}
    fila |= {s["stat"]["name"]: s["base_stat"] for s in datos["stats"]}
    return fila


fichas = pd.DataFrame([estadisticas(n) for n in ["bulbasaur", "charmander", "squirtle", "pikachu"]]).set_index("nombre")
numericas = fichas.select_dtypes("number")
fichas["total"] = numericas.sum(axis=1)
print(fichas.sort_values("total", ascending=False).to_string())
assert numericas.shape[1] == 6

                   tipos  hp  attack  defense  special-attack  special-defense  speed  total
nombre                                                                                      
pikachu         electric  35      55       40              50               50     90    320
bulbasaur   grass/poison  45      49       49              65               65     45    318
squirtle           water  44      48       65              50               64     43    314
charmander          fire  39      52       43              60               50     65    309


## 7. Tabla de tipos

Cada tipo de PokéAPI (`/type/{nombre}`) trae sus relaciones de daño: a qué tipos hace el doble, la mitad o ningún daño. Construir con ellas la tabla de multiplicadores entre fuego, agua, planta, eléctrico, tierra y volador (atacante en filas, defensor en columnas) y comprobar algunas relaciones conocidas.

In [7]:
TIPOS = ["fire", "water", "grass", "electric", "ground", "flying"]


def multiplicadores(tipo):
    relaciones = obtener_json(f"https://pokeapi.co/api/v2/type/{tipo}")["damage_relations"]
    factor = {t: 1.0 for t in TIPOS}
    for clave, valor in [("double_damage_to", 2.0), ("half_damage_to", 0.5), ("no_damage_to", 0.0)]:
        for t in relaciones[clave]:
            if t["name"] in factor:
                factor[t["name"]] = valor
    return factor


tabla_tipos = pd.DataFrame({t: multiplicadores(t) for t in TIPOS}).T
print(tabla_tipos.to_string())
assert tabla_tipos.loc["water", "fire"] == 2 and tabla_tipos.loc["electric", "ground"] == 0 and tabla_tipos.loc["ground", "flying"] == 0

          fire  water  grass  electric  ground  flying
fire       0.5    0.5    2.0       1.0     1.0     1.0
water      2.0    0.5    0.5       1.0     2.0     1.0
grass      0.5    2.0    0.5       1.0     2.0     0.5
electric   1.0    2.0    0.5       0.5     0.0     2.0
ground     2.0    1.0    0.5       2.0     1.0     0.0
flying     1.0    1.0    2.0       0.5     1.0     1.0


## 8. Cadenas evolutivas

Las evoluciones de PokéAPI son un árbol: cada eslabón tiene una especie y una lista `evolves_to` con los siguientes. Pedir la especie de Eevee para obtener la URL de su cadena evolutiva y recorrer el árbol con una función recursiva que devuelva todas las ramas (caminos de la raíz a cada hoja). Hacer lo mismo con una cadena lineal, la de Charmander.

In [8]:
def ramas(eslabon):
    nombre = eslabon["species"]["name"]
    if not eslabon["evolves_to"]:
        return [[nombre]]
    return [[nombre, *resto] for siguiente in eslabon["evolves_to"] for resto in ramas(siguiente)]


def cadena_de(especie):
    url = obtener_json(f"https://pokeapi.co/api/v2/pokemon-species/{especie}")["evolution_chain"]["url"]
    return ramas(obtener_json(url)["chain"])


for especie in ["eevee", "charmander"]:
    todas = cadena_de(especie)
    print(f"{especie}: {len(todas)} rama(s)")
    for rama in todas:
        print("   " + " → ".join(rama))
    assert all(rama[0] == especie for rama in todas)
assert len(cadena_de("eevee")) >= 8 and cadena_de("charmander") == [["charmander", "charmeleon", "charizard"]]

eevee: 8 rama(s)
   eevee → vaporeon
   eevee → jolteon
   eevee → flareon
   eevee → espeon
   eevee → umbreon
   eevee → leafeon
   eevee → glaceon
   eevee → sylveon
charmander: 1 rama(s)
   charmander → charmeleon → charizard
